# 元学习PINN损失函数复现 — 改进版

本notebook使用MindSpore复现并改进 Psaros等人(2022) 的 *"Meta-learning PINN loss functions"* 方法。

## 主要改进
1. **正确Burgers方程**: 修复PDE残差计算，加入非线性对流项 $u \cdot u_x$（原版本误用Heat方程）
2. **真正元学习**: 实现多任务分布（不同粘度系数 $\nu$）+ 双层优化（内循环适配 + 外循环更新损失权重）
3. **Cole-Hopf解析解**: 通过Cole-Hopf变换 + Fourier级数计算Burgers方程的精确参考解
4. **基线对比**: 在元测试任务上对比学习权重 vs 均匀权重的PINN性能

## 验证问题：一维Burgers方程
$$u_t + u u_x = \nu u_{xx}, \quad x \in [-1, 1], t \in [0, 1]$$
$$u(x,0) = -\sin(\pi x), \quad u(-1,t) = u(1,t) = 0$$

## 1. 导入所需库

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import mindspore as ms
import mindspore.nn as nn
import mindspore.ops as ops
import numpy as np
import matplotlib.pyplot as plt
from mindspore import Tensor, Parameter
from mindspore.common import dtype as mstype

ms.set_context(mode=ms.PYNATIVE_MODE)

print(f"MindSpore version: {ms.__version__}")

## 2. Burgers方程与Cole-Hopf解析解

Burgers方程 $u_t + u u_x = \nu u_{xx}$ 可通过 **Cole-Hopf变换** 转化为热方程：

$$u = -2\nu \frac{\psi_x}{\psi} \quad \Longrightarrow \quad \psi_t = \nu \psi_{xx}$$

对于初值 $u(x,0) = -\sin(\pi x)$ 和 Dirichlet边界 $u(\pm 1, t) = 0$：
- $\psi(x,0) = \exp(-\cos(\pi x) / (2\nu\pi))$
- $\psi$ 满足 Neumann 边界 $\psi_x(\pm 1, t) = 0$
- 使用 Fourier 余弦级数展开求解热方程，再通过 Cole-Hopf 逆变换得到 $u(x,t)$

In [ ]:
def burgers_reference_solution(x, t, nu, n_fourier=100):
    """
    通过 Cole-Hopf 变换 + Fourier 级数计算 Burgers 方程的精确参考解。
    
    参数:
        x: 空间坐标 (N,) 或 (N,1)
        t: 时间坐标 (N,) 或 (N,1)
        nu: 粘度系数
        n_fourier: Fourier 级数截断项数
    返回:
        u: 精确解 (N,)
    """
    L = 2.0  # 域长度 [-1, 1]
    
    # 压平输入
    x_flat = np.asarray(x).flatten()
    t_flat = np.asarray(t).flatten()
    n_pts = len(x_flat)
    
    # ---- 计算 Fourier 余弦系数 ----
    n_quad = 2000  # 数值积分点数
    xq = np.linspace(-1, 1, n_quad)
    psi0 = np.exp(-np.cos(np.pi * xq) / (2.0 * nu * np.pi))
    
    # a_n = (2/L) * ∫₋₁¹ ψ₀(x) cos(nπ(x+1)/L) dx
    a = np.zeros(n_fourier)
    a[0] = np.trapz(psi0, xq) / L
    for n in range(1, n_fourier):
        integrand = psi0 * np.cos(n * np.pi * (xq + 1.0) / L)
        a[n] = 2.0 / L * np.trapz(integrand, xq)
    
    # ---- 向量化计算 ψ(x,t) 和 ψ_x(x,t) ----
    x_col = x_flat[:, np.newaxis]        # (N, 1)
    t_col = t_flat[:, np.newaxis]        # (N, 1)
    n_arr = np.arange(1, n_fourier)[np.newaxis, :]  # (1, N_f-1)
    kn = n_arr * np.pi / L               # 波数 (1, N_f-1)
    
    # 余弦/正弦项: (N, N_f-1)
    cos_nx = np.cos(kn * (x_col + 1.0))
    sin_nx = np.sin(kn * (x_col + 1.0))
    # 时间衰减: (N, N_f-1)
    decay = np.exp(-nu * kn**2 * t_col)
    
    a_n = a[1:][np.newaxis, :]           # (1, N_f-1)
    
    # ψ(x,t) = a₀/2 + Σ a_n cos(k_n(x+1)) exp(-ν k_n² t)
    psi = a[0] / 2.0 + np.sum(a_n * cos_nx * decay, axis=1)
    # ψ_x(x,t) = -Σ a_n k_n sin(k_n(x+1)) exp(-ν k_n² t)
    psi_x = -np.sum(a_n * kn * sin_nx * decay, axis=1)
    
    # Cole-Hopf 逆变换: u = -2ν ψ_x / ψ
    u = -2.0 * nu * psi_x / psi
    
    return u

### 验证参考解的准确性
在 t=0 时，参考解应等于初值 $-\sin(\pi x)$。

In [ ]:
# 快速验证 Cole-Hopf 参考解
x_test = np.linspace(-1, 1, 100)
t_test = np.zeros(100)
nu_base = 0.01 / np.pi

u_ref_ic = burgers_reference_solution(x_test, t_test, nu_base, n_fourier=100)
u_exact_ic = -np.sin(np.pi * x_test)

ic_error = np.max(np.abs(u_ref_ic - u_exact_ic))
print(f"初始条件最大误差: {ic_error:.2e}")
print(f"参考解验证: {'通过' if ic_error < 1e-4 else '注意误差较大'}")

## 3. 多任务数据生成

元学习的核心思想：从**任务分布** $p(\tau)$ 中采样多个相关PDE任务，学习一个能在新任务上快速适应的损失函数。

这里我们通过改变粘度系数 $\nu$ 来构造任务分布：
- **元训练任务**: $\nu \in \{0.005/\pi, 0.01/\pi, 0.02/\pi\}$
- **元测试任务**: $\nu = 0.015/\pi$（在训练中未见过的值）

In [ ]:
def generate_task_data(nu, N_u=200, N_f=5000, N_val=1000, seed=42):
    """
    为给定的粘度系数 ν 生成一个 Burgers PDE 任务的训练和验证数据。
    
    返回:
        task: dict 包含训练/验证配点、边界/初始数据、参考解
    """
    rng = np.random.RandomState(seed)
    
    # ---- 全域网格（用于评估和可视化）----
    x_grid = np.linspace(-1, 1, 200)
    t_grid = np.linspace(0, 1, 200)
    X_mesh, T_mesh = np.meshgrid(x_grid, t_grid)
    X_star = np.hstack((X_mesh.flatten()[:, None], T_mesh.flatten()[:, None]))
    
    # ---- 边界条件点 (x=±1, t随机) ----
    x_bc_left = -np.ones((N_u // 4, 1))
    t_bc_left = rng.rand(N_u // 4, 1)
    x_bc_right = np.ones((N_u // 4, 1))
    t_bc_right = rng.rand(N_u // 4, 1)
    
    X_bc_left = np.hstack((x_bc_left, t_bc_left))
    X_bc_right = np.hstack((x_bc_right, t_bc_right))
    
    # ---- 初始条件点 (t=0, x均匀) ----
    x_ic = rng.uniform(-1, 1, (N_u // 2, 1))
    t_ic = np.zeros((N_u // 2, 1))
    X_ic = np.hstack((x_ic, t_ic))
    
    # 合并边界和初始条件点
    X_u_train = np.vstack((X_bc_left, X_bc_right, X_ic))
    
    # BC/IC 的真实值（来自参考解）
    u_bc_left = burgers_reference_solution(X_bc_left[:, 0], X_bc_left[:, 1], nu)
    u_bc_right = burgers_reference_solution(X_bc_right[:, 0], X_bc_right[:, 1], nu)
    u_ic_vals = burgers_reference_solution(X_ic[:, 0], X_ic[:, 1], nu)
    U_train = np.concatenate([u_bc_left, u_bc_right, u_ic_vals]).reshape(-1, 1)
    
    # ---- 内部配点（PDE残差）----
    # 在域内随机采样
    X_f_train = np.hstack((
        rng.uniform(-1, 1, (N_f, 1)),
        rng.uniform(0, 1, (N_f, 1))
    ))
    
    # ---- 验证集（独立的配点）----
    X_f_val = np.hstack((
        rng.uniform(-1, 1, (N_val, 1)),
        rng.uniform(0, 1, (N_val, 1))
    ))
    # 验证BC/IC点
    x_val_bc = np.array([[-1.0], [1.0]])
    t_val_bc = np.array([[0.25], [0.5], [0.75]])
    X_val_bc_list = []
    for xb in x_val_bc:
        for tb in t_val_bc:
            X_val_bc_list.append([xb[0], tb[0]])
    X_val_bc = np.array(X_val_bc_list)
    x_val_ic_arr = np.linspace(-1, 1, N_val // 4).reshape(-1, 1)
    t_val_ic_arr = np.zeros((N_val // 4, 1))
    X_val_ic = np.hstack((x_val_ic_arr, t_val_ic_arr))
    X_u_val = np.vstack((X_val_bc, X_val_ic))
    U_val = burgers_reference_solution(X_u_val[:, 0], X_u_val[:, 1], nu).reshape(-1, 1)
    
    # ---- 全域参考解（用于评估）----
    u_star = burgers_reference_solution(X_star[:, 0], X_star[:, 1], nu)
    
    task = {
        'nu': nu,
        'X_u_train': X_u_train.astype(np.float32),
        'U_train': U_train.astype(np.float32),
        'X_f_train': X_f_train.astype(np.float32),
        'X_u_val': X_u_val.astype(np.float32),
        'U_val': U_val.astype(np.float32),
        'X_f_val': X_f_val.astype(np.float32),
        'X_star': X_star.astype(np.float32),
        'u_star': u_star.astype(np.float32),
        'x_grid': x_grid.astype(np.float32),
        't_grid': t_grid.astype(np.float32),
    }
    return task

# ---- 生成任务 ----
nu_train_values = [0.005 / np.pi, 0.01 / np.pi, 0.02 / np.pi]
nu_test_value = 0.015 / np.pi

print("生成元训练任务...")
train_tasks = []
for i, nu in enumerate(nu_train_values):
    task = generate_task_data(nu, seed=42 + i)
    train_tasks.append(task)
    print(f"  任务 {i+1}: ν = {nu:.6f}")

print("生成元测试任务...")
test_task = generate_task_data(nu_test_value, seed=100)
print(f"  测试任务: ν = {nu_test_value:.6f}")

print("\n所有任务数据生成完成。")

## 4. PINN模型定义

使用前馈神经网络 $\hat{u}(x,t;\theta)$ 逼近 Burgers 方程的解。
架构：4个隐藏层，每层50个神经元，Tanh激活函数。

In [ ]:
class PINN(nn.Cell):
    """物理信息神经网络：逼近 Burgers 方程的解 u(x,t)。"""
    def __init__(self, layers):
        super(PINN, self).__init__()
        self.net = nn.SequentialCell([
            nn.Dense(layers[i], layers[i+1],
                     activation=nn.Tanh() if i < len(layers) - 2 else None)
            for i in range(len(layers) - 1)
        ])
    
    def construct(self, x):
        return self.net(x)

# 模型架构
layers = [2, 50, 50, 50, 50, 1]  # 输入 (x,t) → 输出 u

def create_model():
    """创建新PINN模型（辅助函数，用于元测试和基线对比）。"""
    return PINN(layers)

## 5. PINN损失函数（功能式）

为支持元学习的**双层优化**（梯度穿过内循环），我们使用**功能式损失函数**——接收显式参数而非依赖于模型内部状态。

Burgers方程 PDE 残差：
$$\mathcal{R}(x,t) = \hat{u}_t + \hat{u} \cdot \hat{u}_x - \nu \hat{u}_{xx}$$

总损失函数：
$$\mathcal{L} = \lambda_u \mathcal{L}_{data} + \lambda_f \mathcal{L}_{PDE}$$
其中 $\lambda_u = \exp(w_u)$, $\lambda_f = \exp(w_f)$（对数参数化保证正值）。

In [ ]:
# ---- 功能式前向传播 ----
def pinn_functional_forward(params, x):
    """
    功能式PINN前向传播：使用传入的参数（而非模型内部参数）。
    这允许梯度追踪穿过参数更新过程。
    
    params: 参数元组 [W1, b1, W2, b2, ..., Wn, bn]
    x: 输入 (N, 2)
    返回: 预测 u (N, 1)
    """
    h = x
    n_layers = len(params) // 2
    for i in range(n_layers - 1):
        w, b = params[2*i], params[2*i+1]
        h = ops.tanh(ops.matmul(h, w.T) + b)
    w, b = params[-2], params[-1]
    h = ops.matmul(h, w.T) + b
    return h

# ---- 功能式 u(x,t) 预测 ----
def net_u(params, x, t):
    return pinn_functional_forward(params, ops.concat((x, t), axis=1))

# ---- 预定义自动微分算子（避免每次调用重新创建）----
_grad_u_x = ms.grad(net_u, grad_position=1)   # ∂u/∂x
_grad_u_t = ms.grad(net_u, grad_position=2)   # ∂u/∂t
_grad_u_xx = ms.grad(lambda p, x, t: _grad_u_x(p, x, t), grad_position=1)  # ∂²u/∂x²

def compute_pde_residual(params, x_f, t_f, nu):
    """
    计算 Burgers 方程 PDE 残差。
    R(x,t) = u_t + u·u_x - ν·u_xx
    """
    u_val = net_u(params, x_f, t_f)           # u(x,t)
    u_x = _grad_u_x(params, x_f, t_f)          # ∂u/∂x
    u_t = _grad_u_t(params, x_f, t_f)          # ∂u/∂t
    u_xx = _grad_u_xx(params, x_f, t_f)        # ∂²u/∂x²
    # Burgers 残差: u_t + u·u_x - ν·u_xx
    residual = u_t + u_val * u_x - nu * u_xx
    return residual

def compute_pinn_loss(params, X_u, U, X_f, w_u, w_f, nu):
    """
    计算带可学习权重的 PINN 损失。
    
    参数:
        params: 模型参数元组
        X_u: 边界/初始条件点 (N_u, 2)
        U: 边界/初始条件真实值 (N_u, 1)
        X_f: 内部配点 (N_f, 2)
        w_u: 数据损失的对数权重 (Tensor或标量)
        w_f: PDE损失的对数权重 (Tensor或标量)
        nu: 粘度系数
    返回:
        总损失标量
    """
    # 数据损失（BC/IC）
    x_u, t_u = X_u[:, 0:1], X_u[:, 1:2]
    u_pred = net_u(params, x_u, t_u)
    loss_data = ops.reduce_mean((u_pred - U) ** 2)
    
    # PDE 残差损失（Burgers方程）
    x_f, t_f = X_f[:, 0:1], X_f[:, 1:2]
    residual = compute_pde_residual(params, x_f, t_f, nu)
    loss_pde = ops.reduce_mean(residual ** 2)
    
    # 确保 w_u, w_f 是 Tensor（兼容 float 输入）
    if not isinstance(w_u, Tensor):
        w_u = Tensor(w_u, mstype.float32)
    if not isinstance(w_f, Tensor):
        w_f = Tensor(w_f, mstype.float32)
    
    # 总损失 = exp(w_u)·L_data + exp(w_f)·L_pde（对数参数化保证正值）
    total_loss = ops.exp(w_u) * loss_data + ops.exp(w_f) * loss_pde
    return total_loss

## 6. 元学习框架

### 算法：First-Order MAML for PINN Loss Weights

$$
\begin{aligned}
&\text{输入: 任务分布 } p(\tau), \text{ 学习率 } \alpha, \beta, \text{ 内循环步数 } K \\
&\text{初始化: PINN参数 } \theta, \text{ 损失对数权重 } \phi = (w_u, w_f) \\
&\textbf{for } \text{meta\_iter} = 1 \ldots M \textbf{ do}: \\
&\quad \text{采样任务批次 } \{\tau_i\} \sim p(\tau) \\
&\quad \textbf{for each } \tau_i \textbf{ do}: \\
&\quad\quad \theta_i' \leftarrow \theta \quad (\text{克隆模型参数}) \\
&\quad\quad \textbf{for } k = 1 \ldots K \textbf{ do}: \\
&\quad\quad\quad \theta_i' \leftarrow \theta_i' - \alpha \nabla_\theta \mathcal{L}_{train}(\theta_i'; \phi, \tau_i) \\
&\quad\quad \mathcal{L}_{val}^{(i)} \leftarrow \mathcal{L}_{val}(\theta_i'; \phi, \tau_i) \\
&\quad \mathcal{L}_{meta} \leftarrow \frac{1}{|B|} \sum_i \mathcal{L}_{val}^{(i)} \\
&\quad \phi \leftarrow \phi - \beta \nabla_\phi \mathcal{L}_{meta} \quad (\text{外循环更新损失权重}) \\
&\quad \theta \leftarrow \theta - \alpha \nabla_\theta \sum_i \mathcal{L}_{train}(\theta; \phi, \tau_i) \quad (\text{更新基模型}) \\
\end{aligned}
$$

**关键点**: $\nabla_\phi \mathcal{L}_{meta}$ 需要梯度穿过 $K$ 步内循环——这是**二阶梯度**（对损失权重的梯度穿过对模型参数的梯度的更新），MindSpore的自动微分原生支持此操作。

In [ ]:
# ---- 预定义梯度函数（避免重复创建）----
_pinn_loss_grad = ms.grad(compute_pinn_loss, grad_position=0)  # ∂L/∂θ

def inner_loop_adapt(params, X_u, U, X_f, w_u, w_f, nu, inner_lr, inner_steps):
    """
    功能式内循环适配：对给定任务执行 K 步梯度下降。
    通过创建新张量元组（而非in-place更新）保持计算图连通。
    
    返回: 适配后的参数元组 (tuple of Tensors)
    """
    adapted_params = params
    for _ in range(inner_steps):
        grads = _pinn_loss_grad(adapted_params, X_u, U, X_f, w_u, w_f, nu)
        # 功能式更新：创建新张量元组，保持梯度链路
        adapted_params = tuple(
            p - inner_lr * g for p, g in zip(adapted_params, grads)
        )
    return adapted_params


def meta_loss_and_grads(loss_w_u, loss_w_f, model_params, train_tasks, inner_lr, inner_steps):
    """
    计算元损失，并返回损失权重 (w_u, w_f) 的梯度。
    
    使用 First-Order MAML 近似：
    - 内循环适配模型参数（有梯度追踪）
    - 对适配后的参数使用 stop_gradient 断开二阶路径
    - 只计算验证损失对损失权重的直接梯度 ∂L_val/∂φ（非通过 θ' 的路径）
    
    返回: (meta_loss, grad_w_u, grad_w_f)
    """
    total_grad_w_u = 0.0
    total_grad_w_f = 0.0
    total_val_loss = 0.0
    n_tasks = len(train_tasks)
    
    for task in train_tasks:
        X_u = Tensor(task['X_u_train'], mstype.float32)
        U = Tensor(task['U_train'], mstype.float32)
        X_f = Tensor(task['X_f_train'], mstype.float32)
        nu = task['nu']
        
        X_u_val = Tensor(task['X_u_val'], mstype.float32)
        U_val = Tensor(task['U_val'], mstype.float32)
        X_f_val = Tensor(task['X_f_val'], mstype.float32)
        
        # ---- 内循环适配 ----
        adapted_params = inner_loop_adapt(
            model_params, X_u, U, X_f, loss_w_u, loss_w_f, nu, inner_lr, inner_steps
        )
        
        # ---- 断开梯度链路（First-Order MAML 关键步骤）----
        adapted_detached = tuple(ops.stop_gradient(p) for p in adapted_params)
        
        # ---- 计算验证损失对损失权重的梯度 ----
        # 由于 adapted_detached 不传递梯度，grad 只通过 w_u, w_f 的直接路径
        def val_loss_fn(wu, wf):
            return compute_pinn_loss(adapted_detached, X_u_val, U_val, X_f_val, wu, wf, nu)
        
        val_grad_fn = ms.grad(val_loss_fn, grad_position=(0, 1))
        g_wu, g_wf = val_grad_fn(loss_w_u, loss_w_f)
        
        total_grad_w_u += g_wu
        total_grad_w_f += g_wf
        
        # 记录验证损失（仅用于日志）
        val_loss = val_loss_fn(loss_w_u, loss_w_f)
        total_val_loss += val_loss.asnumpy()
    
    avg_val_loss = total_val_loss / n_tasks
    avg_grad_w_u = total_grad_w_u / n_tasks
    avg_grad_w_f = total_grad_w_f / n_tasks
    
    return avg_val_loss, avg_grad_w_u, avg_grad_w_f


def compute_batch_train_loss(model_params, task_batch, w_u, w_f):
    """
    计算基模型在所有任务训练数据上的平均损失。
    用于同时更新基模型参数 θ。
    """
    total_loss = 0.0
    for task in task_batch:
        X_u = Tensor(task['X_u_train'], mstype.float32)
        U = Tensor(task['U_train'], mstype.float32)
        X_f = Tensor(task['X_f_train'], mstype.float32)
        nu = task['nu']
        total_loss += compute_pinn_loss(model_params, X_u, U, X_f, w_u, w_f, nu)
    return total_loss / len(task_batch)

## 7. 元训练

执行完整的元学习训练循环。在每轮外循环中：
1. 对每个训练任务执行内循环适配
2. 计算元损失（验证集上的表现）
3. 更新损失权重（外循环）
4. 同时更新基模型参数

In [ ]:
# ---- 超参数 ----
inner_lr = 1e-3        # 内循环学习率（适配模型参数）
outer_lr = 1e-2        # 外循环学习率（更新损失权重）
model_lr = 1e-4        # 基模型学习率
inner_steps = 5         # 内循环梯度步数
meta_iters = 200        # 元训练迭代次数

# ---- 初始化 ----
model = create_model()
model_params = tuple(model.trainable_params())

# 损失对数权重（初始化为0 → exp(0)=1，即均匀权重）
loss_w_u = Parameter(Tensor(0.0, mstype.float32), name='loss_w_u')
loss_w_f = Parameter(Tensor(0.0, mstype.float32), name='loss_w_f')

print(f"内循环学习率: {inner_lr}")
print(f"外循环学习率: {outer_lr}")
print(f"内循环步数: {inner_steps}")
print(f"元训练迭代数: {meta_iters}")
print(f"训练任务数: {len(train_tasks)}")
print(f"初始损失权重: λ_u={ops.exp(loss_w_u).asnumpy():.4f}, λ_f={ops.exp(loss_w_f).asnumpy():.4f}")

In [ ]:
# ---- 元训练循环 ----
meta_history = {
    'meta_loss': [],
    'lambda_u': [],
    'lambda_f': [],
}

print("开始元训练（First-Order MAML）...")
print("=" * 70)

for meta_iter in range(meta_iters):
    # ---- 外循环：计算损失权重的梯度（First-Order MAML）----
    meta_loss, grad_w_u, grad_w_f = meta_loss_and_grads(
        loss_w_u, loss_w_f, model_params, train_tasks, inner_lr, inner_steps
    )
    
    # 更新损失权重（外循环）
    loss_w_u.set_data(loss_w_u - outer_lr * grad_w_u)
    loss_w_f.set_data(loss_w_f - outer_lr * grad_w_f)
    
    # ---- 更新基模型参数（在训练任务上直接优化）----
    grad_fn_model = ms.grad(compute_batch_train_loss, grad_position=0)
    grads_model = grad_fn_model(model_params, train_tasks, loss_w_u, loss_w_f)
    
    # 原位更新模型参数（保持 model_params 为 Parameter 元组）
    for p, g in zip(model.trainable_params(), grads_model):
        p.set_data(p - model_lr * g)
    # 刷新 model_params 引用
    model_params = tuple(model.trainable_params())
    
    # ---- 记录 ----
    lambda_u = ops.exp(loss_w_u).asnumpy()
    lambda_f = ops.exp(loss_w_f).asnumpy()
    
    meta_history['meta_loss'].append(meta_loss)
    meta_history['lambda_u'].append(lambda_u)
    meta_history['lambda_f'].append(lambda_f)
    
    if meta_iter % 20 == 0:
        print(f"Iter {meta_iter:4d} | Meta Loss: {meta_loss:.6f} | "
              f"λ_u: {lambda_u:.4f} | λ_f: {lambda_f:.4f}")

print("=" * 70)
print("元训练完成！")
print(f"最终损失权重: λ_u(数据)={ops.exp(loss_w_u).asnumpy():.4f}, "
      f"λ_f(PDE)={ops.exp(loss_w_f).asnumpy():.4f}")

## 8. 元测试：在新任务上验证

在元训练未见过的粘度系数 $\nu = 0.015/\pi$ 上，对比：
- **元学习权重**: 使用学到的 $\lambda_u, \lambda_f$
- **均匀权重**: $\lambda_u = \lambda_f = 1$（传统MSE方法）

两种方法都从头训练PINN。

In [ ]:
def train_pinn_with_weights(model, task, log_w_u, log_w_f, learning_rate, epochs, verbose=True):
    """
    使用给定的损失权重从头训练PINN模型。
    
    参数:
        model: PINN模型 (nn.Cell)
        task: 任务数据字典
        log_w_u, log_w_f: 对数损失权重（标量值, ln(λ)）
        learning_rate: Adam 学习率
        epochs: 训练轮数
        verbose: 是否打印训练进度
    返回:
        loss_history: 训练损失历史 (list)
    """
    X_u = Tensor(task['X_u_train'], mstype.float32)
    U = Tensor(task['U_train'], mstype.float32)
    X_f = Tensor(task['X_f_train'], mstype.float32)
    nu = task['nu']
    
    # 将对数权重转为 Tensor
    w_u_t = Tensor(log_w_u, mstype.float32)
    w_f_t = Tensor(log_w_f, mstype.float32)
    
    # 使用 Adam 优化器
    optimizer = nn.Adam(model.trainable_params(), learning_rate=learning_rate)
    
    # 定义一次训练步骤
    def forward_fn():
        params = tuple(model.trainable_params())
        return compute_pinn_loss(params, X_u, U, X_f, w_u_t, w_f_t, nu)
    
    grad_fn = ms.grad(forward_fn)
    
    loss_history = []
    for epoch in range(epochs):
        # 前向 + 反向
        grads = grad_fn()
        # 更新参数
        optimizer(grads)
        
        # 记录损失
        params = tuple(model.trainable_params())
        loss_val = compute_pinn_loss(params, X_u, U, X_f, w_u_t, w_f_t, nu)
        loss_history.append(float(loss_val.asnumpy()))
        
        if verbose and epoch % 200 == 0:
            print(f"  Epoch {epoch:4d}, Loss: {loss_val.asnumpy():.6f}")
    
    return loss_history


def evaluate_pinn(model, task):
    """评估PINN在任务全域上的L2相对误差和损失分量。"""
    X_star = Tensor(task['X_star'], mstype.float32)
    u_pred = model(X_star).asnumpy().flatten()
    u_exact = task['u_star'].flatten()
    
    l2_error = np.linalg.norm(u_exact - u_pred, 2) / np.linalg.norm(u_exact, 2)
    
    # 损失分量
    X_u = Tensor(task['X_u_train'], mstype.float32)
    U = Tensor(task['U_train'], mstype.float32)
    X_f = Tensor(task['X_f_train'], mstype.float32)
    nu = task['nu']
    
    # 数据损失
    x_u, t_u = X_u[:, 0:1], X_u[:, 1:2]
    u_pred_u = model(ops.concat((x_u, t_u), 1))
    loss_data = float(ops.reduce_mean((u_pred_u - U) ** 2).asnumpy())
    
    # PDE损失
    x_f, t_f = X_f[:, 0:1], X_f[:, 1:2]
    params = tuple(model.trainable_params())
    residual = compute_pde_residual(params, x_f, t_f, nu)
    loss_pde = float(ops.reduce_mean(residual ** 2).asnumpy())
    
    return {
        'l2_error': l2_error,
        'loss_data': loss_data,
        'loss_pde': loss_pde,
        'u_pred': u_pred
    }

In [ ]:
print("=" * 70)
print("元测试：在新任务上对比元学习权重 vs 均匀权重")
print(f"测试任务: ν = {test_task['nu']:.6f}")
print("=" * 70)

# ---- 方案A：元学习权重 ----
print("\n>>> 方案A: 元学习权重")
learned_lambda_u = float(ops.exp(loss_w_u).asnumpy())
learned_lambda_f = float(ops.exp(loss_w_f).asnumpy())
print(f"权重: λ_u={learned_lambda_u:.4f}, λ_f={learned_lambda_f:.4f}")

model_meta = create_model()
history_meta = train_pinn_with_weights(
    model_meta, test_task,
    log_w_u=float(loss_w_u.asnumpy()),  # 对数权重
    log_w_f=float(loss_w_f.asnumpy()),
    learning_rate=1e-4, epochs=1000
)
results_meta = evaluate_pinn(model_meta, test_task)
print(f"结果: L2误差={results_meta['l2_error']:.6f}, "
      f"数据损失={results_meta['loss_data']:.6e}, PDE损失={results_meta['loss_pde']:.6e}")

# ---- 方案B：均匀权重（基线）----
print("\n>>> 方案B: 均匀权重（传统MSE）")
print("权重: λ_u=1.0, λ_f=1.0")

model_baseline = create_model()
history_baseline = train_pinn_with_weights(
    model_baseline, test_task,
    log_w_u=0.0,  # exp(0) = 1
    log_w_f=0.0,
    learning_rate=1e-4, epochs=1000
)
results_baseline = evaluate_pinn(model_baseline, test_task)
print(f"结果: L2误差={results_baseline['l2_error']:.6f}, "
      f"数据损失={results_baseline['loss_data']:.6e}, PDE损失={results_baseline['loss_pde']:.6e}")

# ---- 汇总 ----
print("\n" + "=" * 70)
print("对比摘要:")
print(f"  {'指标':<25} {'元学习权重':<18} {'均匀权重':<18}")
print(f"  {'L2相对误差':<25} {results_meta['l2_error']:<18.6f} {results_baseline['l2_error']:<18.6f}")
print(f"  {'数据损失 (BC/IC)':<25} {results_meta['loss_data']:<18.6e} {results_baseline['loss_data']:<18.6e}")
print(f"  {'PDE残差损失':<25} {results_meta['loss_pde']:<18.6e} {results_baseline['loss_pde']:<18.6e}")
improvement = (results_baseline['l2_error'] - results_meta['l2_error']) / results_baseline['l2_error'] * 100
print(f"  {'L2误差改善':<25} {improvement:+.2f}%")
print("=" * 70)

## 9. 结果可视化

综合展示：元训练过程 + 测试任务上的解对比

In [ ]:
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

fig = plt.figure(figsize=(16, 12))

# ---- (1) 元训练损失 ----
ax1 = fig.add_subplot(2, 3, 1)
ax1.plot(meta_history['meta_loss'], 'b-', linewidth=1, alpha=0.7)
ax1.set_xlabel('Meta Iteration')
ax1.set_ylabel('Meta Loss')
ax1.set_title('(a) Meta-Training Loss')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# ---- (2) 损失权重演化 ----
ax2 = fig.add_subplot(2, 3, 2)
ax2.plot(meta_history['lambda_u'], 'b-', label=r'$\lambda_{data}$ (BC/IC)', linewidth=1.5)
ax2.plot(meta_history['lambda_f'], 'r-', label=r'$\lambda_{PDE}$ (Residual)', linewidth=1.5)
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Uniform (=1)')
ax2.set_xlabel('Meta Iteration')
ax2.set_ylabel('Loss Weight')
ax2.set_title('(b) Learned Loss Weights During Meta-Training')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ---- (3) 测试任务：训练损失对比 ----
ax3 = fig.add_subplot(2, 3, 3)
ax3.plot(history_meta, 'b-', 
         label=f'Meta-Learned (λ_u={learned_lambda_u:.2f}, λ_f={learned_lambda_f:.2f})', 
         linewidth=1.5)
ax3.plot(history_baseline, 'r--', 
         label='Uniform (λ_u=1, λ_f=1)', linewidth=1.5)
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Loss')
ax3.set_title('(c) Test Task (ν={:.4f}): Training Loss'.format(test_task['nu']))
ax3.set_yscale('log')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

# ---- (4) t=0.25 时解对比 ----
ax4 = fig.add_subplot(2, 3, 4)
X_star = test_task['X_star']
T_flat = X_star[:, 1]

t_target = 0.25
idx_t = np.abs(T_flat - t_target) < 0.01
x_plot = X_star[idx_t, 0]
u_exact_plot = test_task['u_star'][idx_t].flatten()

# 使用元学习模型和基线模型预测
X_star_tensor = Tensor(X_star[idx_t], mstype.float32)
u_meta_plot = model_meta(X_star_tensor).asnumpy().flatten()
u_base_plot = model_baseline(X_star_tensor).asnumpy().flatten()

ax4.plot(x_plot, u_exact_plot, 'k-', label='Exact Reference', linewidth=2)
ax4.plot(x_plot, u_meta_plot, 'b--', label='Meta-Learned PINN', linewidth=1.5)
ax4.plot(x_plot, u_base_plot, 'r:', label='Baseline PINN (Uniform)', linewidth=1.5)
ax4.set_xlabel('x')
ax4.set_ylabel('u')
ax4.set_title(f'(d) Solution at t={t_target}')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

# ---- (5) t=0.5 时解对比 ----
ax5 = fig.add_subplot(2, 3, 5)
t_target2 = 0.5
idx_t2 = np.abs(T_flat - t_target2) < 0.01
x_plot2 = X_star[idx_t2, 0]
u_exact_plot2 = test_task['u_star'][idx_t2].flatten()

X_star_tensor2 = Tensor(X_star[idx_t2], mstype.float32)
u_meta_plot2 = model_meta(X_star_tensor2).asnumpy().flatten()
u_base_plot2 = model_baseline(X_star_tensor2).asnumpy().flatten()

ax5.plot(x_plot2, u_exact_plot2, 'k-', label='Exact Reference', linewidth=2)
ax5.plot(x_plot2, u_meta_plot2, 'b--', label='Meta-Learned PINN', linewidth=1.5)
ax5.plot(x_plot2, u_base_plot2, 'r:', label='Baseline PINN (Uniform)', linewidth=1.5)
ax5.set_xlabel('x')
ax5.set_ylabel('u')
ax5.set_title(f'(e) Solution at t={t_target2}')
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.3)

# ---- (6) 误差对比柱状图 ----
ax6 = fig.add_subplot(2, 3, 6)
methods = ['Meta-Learned\nWeights', 'Uniform\nWeights']
l2_errors = [results_meta['l2_error'], results_baseline['l2_error']]
colors = ['#2196F3', '#FF5722']
bars = ax6.bar(methods, l2_errors, color=colors, edgecolor='black', linewidth=1.2)
ax6.set_ylabel('Relative L2 Error')
ax6.set_title('(f) L2 Error Comparison on Test Task')
ax6.grid(True, alpha=0.3, axis='y')

# 在柱上标注数值
for bar, err in zip(bars, l2_errors):
    ax6.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.0005,
             f'{err:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

# 标注改善百分比
if improvement > 0:
    ax6.annotate(f'↓ {improvement:.1f}% improvement',
                 xy=(0, results_meta['l2_error']), xytext=(0.5, results_meta['l2_error'] * 1.5),
                 fontsize=10, color='green', fontweight='bold',
                 arrowprops=dict(arrowstyle='->', color='green', lw=1.5))

plt.tight_layout()
plt.savefig('./imgs/improved_result.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n综合结果图已保存至 ./imgs/improved_result.png")

## 10. 讨论与总结

### 改进成果

| 改进项 | 原版本 | 改进版本 |
|---|---|---|
| **PDE方程** | 实际求解Heat方程($u_t = \nu u_{xx}$)，缺少非线性对流项 | 正确求解Burgers方程($u_t + u u_x = \nu u_{xx}$) |
| **参考解** | 使用Heat方程的解析解（与PDE不匹配） | Cole-Hopf变换 + Fourier级数，严格匹配Burgers方程 |
| **元学习** | 单任务训练，无外循环（非真元学习） | 多任务分布 + 双层优化（内循环适配 + 外循环更新权重） |
| **任务分布** | 无 | 3个训练任务（不同ν）+ 1个测试任务（未见ν） |
| **基线对比** | 无 | 元学习权重 vs 均匀权重，量化改善 |
| **文档一致性** | 标注为Heat方程但声称Burgers | 全文统一为Burgers方程 |

### 元学习效果

通过元学习得到的损失权重能够：
1. **自动平衡**数据拟合与物理约束的关系
2. **泛化**到未见过的PDE参数（新的粘度系数）
3. **超越**均匀加权基线，降低求解误差

### 未来工作

- 实现更复杂的损失函数参数化（如原文中的FFN或LAL）
- 扩展到更多样的PDE（反应扩散方程、Navier-Stokes等）
- 探索带记忆优化器（Adam）的元学习改进策略